# Queen Editor — Repo çekimi (Bölüm 1)

Bu notebook private `Internal-tools` reposunu Colab'a klonlar — Queen Editor'ün ilk adımı:
**klon çalışıyor mu?** Sunucu, arayüz, Drive, ComfyUI **yok**; sadece kodun Colab'a indiğini kanıtlar.

## Kullanım
1. Bu `app.ipynb`'yi Colab'a yükle (**File → Upload notebook**).
2. Aşağıdaki **CONFIG** hücresine GitHub token'ını yapıştır (fine-grained, yalnız bu repo,
   `Contents: read` — kurulum için `README.md`).
3. **Runtime → Run all.**
4. En alttaki çıktıda `queen-editor/` içeriği + commit hash görünmeli; **token görünmemeli**.

> Token'ı yapıştırdıktan sonra notebook'u bu haliyle **kaydedip commit'leme** — token sızar.
> `GITHUB_TOKEN`'ı boş bırak.

In [ ]:
# === CONFIG ===
# Fine-grained GitHub token, this repo only, "Contents: read" (see README for setup).
# Paste it here at runtime; leave it empty ("") before saving/committing -- the token grants
# repo access and must never land in git history.
GITHUB_TOKEN = ""

BRANCH    = "feat/queen-editor-v1"       # dev branch for now; switch to "main" after merge
REPO      = "AltanBaysal/Internal-tools" # <owner>/<repo>
CLONE_DIR = "/content/Internal-tools"    # clone target on Colab's local disk

assert GITHUB_TOKEN, "❌ GITHUB_TOKEN boş — CONFIG hücresine fine-grained token'ını yapıştır (README'ye bak)"
print("✓ CONFIG hazır")
print(f"✓ Dal: {BRANCH}  |  Repo: {REPO}  |  Hedef: {CLONE_DIR}")

In [ ]:
# === Clone (delete-and-reclone: the local tree is disposable, always fetch the latest) ===
# subprocess.run with an argument LIST (not shell=True): the token never reaches the shell
# history or a log line. On failure git's stderr is printed RAW, with the token masked.
import os, shutil, subprocess

def _mask(text):
    """Replace the token with <token> so no output ever carries it."""
    return text.replace(GITHUB_TOKEN, "<token>") if GITHUB_TOKEN else text

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)              # no pull/merge -- a fresh clone has one behaviour

clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"   # never printed (carries the token)
result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, CLONE_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    # Raw git output, token masked -- never invent a cause (repo comment rule).
    raise RuntimeError("❌ Klon başarısız:\n" + _mask(result.stderr.strip() or result.stdout.strip()))

print("✓ Klon tamam")

In [ ]:
# === Verify: prove the right branch/commit landed, without leaking anything ===
import os, subprocess

qe_dir = os.path.join(CLONE_DIR, "queen-editor")
assert os.path.isdir(qe_dir), f"❌ {qe_dir} yok — yanlış dal ya da eksik dosya (klon queen-editor/ getirmedi)"
assert os.path.exists(os.path.join(qe_dir, "app.ipynb")), \
    "❌ queen-editor/app.ipynb yok — klon beklenen içeriği getirmedi"

commit = subprocess.run(
    ["git", "-C", CLONE_DIR, "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True,
).stdout.strip()

print(f"✓ Klonlanan commit: {commit}  (dal: {BRANCH})")
print("✓ queen-editor/ içeriği:")
for name in sorted(os.listdir(qe_dir)):
    print(f"   - {name}")